# HVAC Control with Reinforcement Learning - Colab Quick Start

This notebook helps you set up and run the HVAC-RL project on Google Colab.

**Requirements:**
- Google Colab with GPU (T4 or better for LLM inference)
- ~10GB disk space

**Steps:**
1. Clone the repository
2. Install dependencies
3. Set up the BEAR environment
4. Run a quick test

## Step 1: Check GPU and Clone Repository

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone the repository
import os

REPO_URL = "https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git"
REPO_DIR = "/content/HVAC-RL"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 2: Install Dependencies

In [ ]:
# Install all dependencies
!pip install -q -r requirements.txt

# Install the BEAR package in development mode
!pip install -q -e .

print("Dependencies installed!")

In [ ]:
# Verify installations
import gymnasium as gym
import stable_baselines3 as sb3
import transformers
import pvlib

print(f"gymnasium: {gym.__version__}")
print(f"stable_baselines3: {sb3.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"pvlib: {pvlib.__version__}")

## Step 3: Set Up Data Directory

The BEAR simulator needs building data files. We'll set up the correct paths.

In [ ]:
import os
import shutil

# Check Data directory location
data_dir = os.path.join(REPO_DIR, "Data")
bear_data_dir = os.path.join(REPO_DIR, "BEAR", "Data")

# Create symlink if needed
if os.path.exists(data_dir) and not os.path.exists(bear_data_dir):
    os.symlink(data_dir, bear_data_dir)
    print(f"Created symlink: {bear_data_dir} -> {data_dir}")

# List available data files
print("\nAvailable building data files:")
for f in os.listdir(data_dir):
    print(f"  - {f}")

## Step 4: Test BEAR Environment

In [ ]:
# Add project to path
import sys
sys.path.insert(0, REPO_DIR)

# Import BEAR modules
from BEAR.Env.env_building import BuildingEnvReal
from BEAR.Utils.utils_building import ParameterGenerator
import numpy as np

print("BEAR modules imported successfully!")

In [ ]:
# Create environment with correct data path
data_root = os.path.join(REPO_DIR, "Data")

# Use OfficeLarge since we have data for it
# Note: File has a special name, we need to handle it
building_file = os.path.join(data_root, "ASHRAE901_OfficeLarge_STD2019_Tucson.table 5.02.27 PM.htm")

if os.path.exists(building_file):
    print(f"Using building file: {building_file}")
    Parameter = ParameterGenerator(
        Building=building_file,  # Use direct file path
        Weather='Hot_Dry',
        Location='Tucson',
        root=data_root
    )
else:
    print("Building file not found, using default parameters")
    Parameter = ParameterGenerator(
        Building='OfficeSmall',
        Weather='Hot_Dry', 
        Location='Tucson',
        root=data_root
    )

# Create environment
env = BuildingEnvReal(Parameter)
print(f"\nEnvironment created successfully!")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Number of rooms: {env.roomnum}")

In [ ]:
# Run a quick simulation
import matplotlib.pyplot as plt

obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")

rewards = []
temperatures = []

# Run for 24 hours
for step in range(24):
    # Random action
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    rewards.append(reward)
    temperatures.append(info['zone_temperature'][:6].mean())
    
    if done:
        break

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rewards)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Reward')
axes[0].set_title('Rewards over 24 hours')

axes[1].plot(temperatures)
axes[1].axhline(y=22, color='r', linestyle='--', label='Target (22C)')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Temperature (C)')
axes[1].set_title('Average Zone Temperature')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nTotal reward: {sum(rewards):.2f}")
print("Environment test completed!")

## Step 5: Train PPO Agent (Optional)

This will train a PPO agent on the HVAC environment.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.utils import set_random_seed

# Reset environment
env.reset()
set_random_seed(42)

# Create PPO model
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
)

print("PPO model created!")

In [ ]:
# Train for a short time (increase total_timesteps for better results)
TRAIN_STEPS = 10000  # Increase to 100000+ for better results

print(f"Training for {TRAIN_STEPS} steps...")
model.learn(total_timesteps=TRAIN_STEPS, progress_bar=True)
print("Training completed!")

# Save model
model.save("ppo_hvac_colab")
print("Model saved to ppo_hvac_colab.zip")

In [ ]:
# Evaluate trained model
obs, _ = env.reset()
rewards = []
temperatures = []

for step in range(24):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    rewards.append(reward)
    temperatures.append(info['zone_temperature'][:6].mean())
    if done:
        break

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rewards)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Reward')
axes[0].set_title('PPO Agent Rewards')

axes[1].plot(temperatures)
axes[1].axhline(y=22, color='r', linestyle='--', label='Target (22C)')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Temperature (C)')
axes[1].set_title('PPO Agent - Zone Temperature')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"PPO Total reward: {sum(rewards):.2f}")

## Step 6: LLM Inference (Optional - Requires more GPU memory)

This section demonstrates LLM-based HVAC control. Requires T4 GPU or better.

In [ ]:
# Check if we have enough GPU memory for LLM
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    if gpu_mem < 15:
        print("Warning: Less than 15GB GPU memory. LLM inference may fail.")
        print("Consider using Colab Pro with A100 or V100 GPU.")
else:
    print("No GPU available. LLM inference will be very slow.")

In [ ]:
# Import LLM agent (only if you have sufficient GPU memory)
from core_modules.llm_agent_colab import load_llm, call_llm, parse_actions

# Test with a smaller model first
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Smaller, faster
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Full model

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes...")

In [ ]:
# Create a sample prompt
sample_prompt = """
You are an HVAC control expert. Analyze the building state and provide optimal HVAC actions.

Current Building State:
- Zone temperatures: [23.5, 24.1, 22.8, 25.0, 21.5, 22.0] Celsius
- Outside temperature: 35.0 Celsius
- Target temperature: 22.0 Celsius
- Time: 14:00 (afternoon)

Provide actions for 6 zones as values between -1 (max cooling) and 1 (max heating).

Format your response as:
Analysis: [your analysis]
Actions: [action1, action2, action3, action4, action5, action6]
"""

# Call LLM
import os
os.environ["MODEL_NAME"] = MODEL_NAME

response = call_llm(sample_prompt, n_actions=6)
print("LLM Response:")
print(response)

# Parse actions
actions, meta = parse_actions(response, n=6)
print(f"\nParsed Actions: {actions}")
print(f"Parse Info: {meta}")

## Troubleshooting

### Common Issues:

1. **ModuleNotFoundError: No module named 'BEAR'**
   - Run `!pip install -e .` in the repository directory
   - Make sure you're in the correct working directory

2. **FileNotFoundError for .htm files**
   - Check that the Data directory exists
   - Use direct file paths instead of building type names

3. **CUDA out of memory**
   - Use a smaller model (Qwen2.5-1.5B-Instruct)
   - Restart runtime and clear GPU memory
   - Use Colab Pro for more GPU memory

4. **Import errors**
   - Restart runtime after installing packages
   - Check that all dependencies are installed

In [ ]:
# Clean up
env.close()
print("Environment closed.")